# 01a · Prepare the four image benchmarks (Dirichlet split + backdoor)
Port of `main_image/prepare_data.py`. For each dataset and seed: 10 clients, Dirichlet(α) label skew
(α=0.1; CIFAR-100 α=1), attacker = `client_3`, 4×4 white patch → label 9 on **all** its samples.
Test sets: clean + triggered (target-class images are left untouched, as in the official code).

* SVHN / CIFAR-10 / CIFAR-100 download automatically. **EuroSAT** is looked up in `data/EuroSAT/2750`
  (or `data/eurosat/2750`); otherwise torchvision's downloader is tried, then you get instructions.
* Disk: roughly 10 GB for everything below (fp32 tensors; estimate from tensor sizes, not measured).
* Data are split once per seed, so seeds 0 and 1 give different partitions (as in the paper's two runs).

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, PROTOCOLS, image_config
from fuguard_repro.data_image import prepare_image_data
from fuguard_repro.paths import ensure_dirs
ensure_dirs()

DATASETS = IMAGE_DATASETS            # e.g. ["CIFAR10"] to start small
SEEDS = (0, 1)
for ds in DATASETS:
    for seed in SEEDS:
        prepare_image_data(image_config(ds), seed)

### 100-client SVHN (Fig. 9 / scalability)
The paper does not state α or the attacker index for this run; we assume the SVHN α=0.1 and `client_3`.
With 100 clients and α=0.1 some clients can be (nearly) empty - the engine skips empty clients and tells you if the attacker is empty.

In [ ]:
for seed in SEEDS:
    prepare_image_data(image_config("SVHN", num_clients=100, tag="SVHN-100"), seed)